In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd

# Import your custom modules
from Model.model import STGCN
from Model.data_utils import scaled_laplacian, cheb_poly_approx, z_score, TrafficDataset, first_approximation

In [ ]:
class Args:
    # Hyperparameters from the original paper
    n_route = 43 # Number of nodes/sensors
    n_his = 12 # Historical time steps (M=12, i.e., 60 mins)
    n_pred = 9 # Prediction time steps (H=9, i.e., 45 mins)
    batch_size = 20
    epoch = 50
    ks = 3 # Spatial kernel size
    kt = 3 # Temporal kernel size
    lr = 1e-3 # Learning rate

    # ST-Conv Block channel configurations [c_in, c_spatial, c_out]
    blocks = [[1, 32, 64], [64, 32, 128]]

args = Args()

# 1. Select Device (Support for Mac MPS, CUDA, or CPU)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

In [ ]:
# 2. Build Graph Adjacency Matrix & Chebyshev Polynomials
print("Building Graph")
W_df = pd.read_csv('../dataset/ShanghaiRailway/W_43_P.csv', header=None) 
W = W_df.values
# W = np.random.rand(args.n_route, args.n_route) 
np.fill_diagonal(W, 0)

if args.ks > 1:
    # Use Chebyshev approximation
    L = scaled_laplacian(W)
    graph_kernel = cheb_poly_approx(L, args.ks, args.n_route).to(device)
else:
    # Use 1st-order approximation
    graph_kernel = first_approximation(W, args.n_route).to(device)

print("Loading and Preprocessing Data")
# This need to replace with actual traffic data of shape [Time, Node, Channel]
V_df = pd.read_csv('../dataset/ShanghaiRailway/V_43.csv', header=None)
v_values = V_df.values
raw_data = np.expand_dims(v_values, axis=-1)

# Split Data (70% train, 10% val, 20% test)
train_len = int(raw_data.shape[0] * 0.7)
val_len = int(raw_data.shape[0] * 0.1)

train_data = raw_data[:train_len]
val_data = raw_data[train_len:train_len+val_len]
test_data = raw_data[train_len+val_len:]

# Calculate Z-score stats based ONLY on training data
stats = {'mean': np.mean(train_data), 'std': np.std(train_data)}

train_data_norm = z_score(train_data, stats['mean'], stats['std'])
val_data_norm = z_score(val_data, stats['mean'], stats['std'])
test_data_norm = z_score(test_data, stats['mean'], stats['std'])

$43 (node) \times 3 (kernel) = 129$

In [ ]:
train_data_norm.shape, graph_kernel.shape, L.shape

In [ ]:
train_dataset = TrafficDataset(train_data_norm, args.n_his)
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)

print("Initializing STGCN Model")
model = STGCN(args.ks, args.kt, args.blocks, args.n_his, args.n_route).to(device)

# Define Optimiser & Scheduler
optimizer = torch.optim.RMSprop(model.parameters(), lr=args.lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.7)

print(model)